## Calculate composite atmospheric jet for El Nino and La Nina states, for looking at the effects on wind shear that can affect the hurricane season in the North Atlantic

Using ERA5 reanalysis.

In [ ]:
# specify True if large data sets are not available 
# (e.g., if they were on Cheyenne and the notebook is now used locally)
large_datasets_are_available=False

In [ ]:
import numpy as np
from numpy import linalg
import matplotlib.pyplot as plt
import matplotlib
from netCDF4 import Dataset
import cartopy.crs as ccrs
import cartopy
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import matplotlib.patches as mpatches
import pickle
import subprocess

# font required by PUP:
plt.rcParams['font.family'] = 'Myriad Pro'

print("done.")

if large_datasets_are_available:
    vars2save=[]
else:
    print("\nextracting variables: ",end="")
    pickle_filename='hurricanes_ElNino_LaNina_jet_composites_ERA5.pickle'
    savedir='../../../Data-for-teaching-staff/Hurricanes/'
    with open(savedir+pickle_filename, 'rb') as file:
            d = pickle.load(file)
            # print information about each extracted variable:
            for key in list(d.keys()):
                print(key,",",end="")
                #print(\"; type=\",type(d[key]))
            globals().update(d)
    print(" done.\n")

# read atmospheric grid information:
if large_datasets_are_available:
    data_dir="/Users/eli/Downloads/"
    file="UV200.nc"
    ncfile = Dataset(data_dir+file, 'r');
    lat = ncfile.variables['latitude'][:]
    lon = ncfile.variables['longitude'][:]
    times=ncfile.variables['time'][:]

    # dont use data from partial year in 2020:
    times=times[:-7]

    # read ocean grid information:
    data_dir="/Users/eli/Downloads/"
    file="SST.nc"
    ncfile_ocean = Dataset(data_dir+file, 'r');
    lat_ocean = ncfile_ocean.variables['latitude'][:]
    lon_ocean = ncfile_ocean.variables['longitude'][:]
    #dates_ocean=ncfile_ocean.variables['date'][:]

    # open SST file:
    data_dir="/Users/eli/Downloads/"
    file="SST.nc"
    ncfile= Dataset(data_dir+file, 'r');
    SST= ncfile.variables['sst']

    # open U,V files:
    data_dir="/Users/eli/Downloads/"
    file="UV200.nc"
    ncfile= Dataset(data_dir+file, 'r');
    U200= ncfile.variables['u']
    V200 = ncfile.variables['v']
    data_dir="/Users/eli/Downloads/"
    file="UV850.nc"
    ncfile= Dataset(data_dir+file, 'r');
    U850= ncfile.variables['u']
    V850 = ncfile.variables['v']

    # get grid/ time limits:
    Nt,Nz,Ny,Nx=U200.shape

    # save for use if large data are not available:
    vars2save.extend([lon,lat,times,Nt,Nz,Ny,Nx])

    
# calculate dates and month for each day in the record: model times are hours since 1900-01-01 00:00:00.0
timestamp0=datetime.timestamp(datetime.fromisoformat('1900-01-01'))+20*60+40
months=np.zeros(len(times))
dates=[]
idate=-1
for time in times:
    idate=idate+1
    date=datetime.fromtimestamp((time*3600+timestamp0))
    dates.append(date)
    month=dates[idate].month
    months[idate]=month
    #print(time,date,month)
print("calculated months time series.")

# initialize monthly climatoloigy arrays:
U200_climatology=np.zeros((12,Ny,Nx))
V200_climatology=np.zeros((12,Ny,Nx))
U850_climatology=np.zeros((12,Ny,Nx))
V850_climatology=np.zeros((12,Ny,Nx))
SST_climatology=np.zeros((12,Ny,Nx))

if large_datasets_are_available:
    print("dimensions of U200,V200:",U200.shape)
    print("dimensions of SST:",SST.shape)
    print("first time:",dates[0],",last time:",dates[-1])

In [ ]:
# plot quiver for first time in data, just for verification:
if large_datasets_are_available:
    projection=ccrs.PlateCarree(central_longitude=0.0)
    fig,axes=plt.subplots(1,1,figsize=(5,3),dpi=300,subplot_kw={'projection': projection})

    # U,V:
    # ----------------
    U=U200[0,0,:,:]
    V=V200[0,0,:,:]
    SST1=SST[0,0,:,:]-273.15
    axes.set_extent([-170, 10, -20, 60], crs=ccrs.PlateCarree())
    axes.coastlines(resolution='110m',linewidth=0.5)
    axes.stock_img()
    axes.gridlines(linewidth=0.25)
    plt.set_cmap('viridis')
    levels=np.arange(0,6.3,0.3)
    c=axes.pcolormesh(lon_ocean,lat_ocean, SST1[:,:])#,levels=levels)
    s=15
    axes.quiver(lon[::s], lat[::s], U[::s,::s], V[::s,::s],scale=1000\
                ,width=0.001,headwidth=10,headlength=10)
    clb=plt.colorbar(c, shrink=0.6, pad=0.02,ax=axes)
    clb.set_label('C')
    axes.set_title('SST, (U,V)')

    # finalize and show plot:
    #plt.subplots_adjust(top=0.92, bottom=0.08, left=0.01, right=0.95, hspace=0.15,wspace=-0.4)
    plt.show();

print("done")

In [ ]:
# define some functions to be able to process all files efficiently without running into memory problems

def extract_data_for_one_century(variable_name,century):
    # extract data for one month only during a single century
    time_range_string="%2.2d0001-%2.2d9912"       % (century,century)
    if variable_name=='SST':
        dir_name="/glade/collections/cdg/data/cesmLE/CESM-CAM5-BGC-LE/ocn/proc/tseries/monthly/"+variable_name+"/"
        file_name="b.e11.B1850C5CN.f09_g16.005.pop.h.SST."+time_range_string+".nc"
    else:
        dir_name="/glade/collections/cdg/data/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/monthly/"+variable_name+"/"
        file_name="b.e11.B1850C5CN.f09_g16.005.cam.h0."+variable_name+"."+time_range_string+".nc"
    ncfile=Dataset(dir_name+file_name, 'r');
    variable =ncfile.variables[variable_name]
    return variable


def calculate_NINO34_timeseries():
    """ calculate a monthly time series of NINO3.4."""
    print("calculate_NINO34_timeseries... ",end="")
    # read data:
    data_dir="/Users/eli/Downloads/"
    file="SST.nc"
    ncfile= Dataset(data_dir+file, 'r');
    SST= ncfile.variables['sst']
    it=-1
    NINO34_timeseries=np.zeros(len(times))
    ilat=np.logical_and(lat<=5,lat>=-5)
    ilon=np.logical_and(lon<=360-120,lon>=360-170)
    if 0:
        plt.pcolormesh(lon,lat,mask)
        plt.colorbar()
        plt.show()
        plt.pause(1000)
    NINO34_timeseries=np.nanmean(SST[:,0,ilat,ilon],axis=(1,2))

    # now remove monthly climatology from the NINO3 time series:
    NINO34_monthly_climatology=np.zeros(12)
    for m in range(12):
        NINO34_monthly_climatology[m]=np.mean(NINO34_timeseries[m::12])
        NINO34_timeseries[m::12]= \
           NINO34_timeseries[m::12]-NINO34_monthly_climatology[m]

    print(" done.")
    return NINO34_timeseries


def calculate_composite(variable,mask_timeseries,remove_monthly_climatology,monthly_climatology):
    first_time_read=True
    iavg=0
    for t in range(len(times)):
        #print(t,",",end="")
        # read data:
        if not np.isnan(mask_timeseries[t]):
            if first_time_read:
                first_time_read=False
                variable_avg=1.0*variable[t,0,:,:]
            else:
                variable_avg=variable_avg+variable[t,0,:,:]
            if remove_monthly_climatology:
                variable_avg=variable_avg-monthly_climatology[int(months[t])-1,:,:]

            iavg=iavg+1

    if iavg>0:
        variable_avg=variable_avg/iavg
        variable_avg[variable_avg.mask]=np.nan
    else:
        print("\n\n\n*** error: no times to composite over.\n\n\n")
    print(" done.")
    
    return variable_avg

print("function defenitions updated.")

In [ ]:
# calculate and plot a time series of averaged total precipitation over california as function of time:
if large_datasets_are_available:
    NINO34_timeseries=calculate_NINO34_timeseries()
    NINO34_timeseries=NINO34_timeseries[0:len(dates)]
    
    # save for use if large data are not available:
    vars2save.extend([NINO34_timeseries])

mean=np.mean(NINO34_timeseries)
std=np.std(NINO34_timeseries)
print("nino3.4 mean, std=",mean,std)
ElNino_threshold=mean+std*1.5
LaNina_threshold=mean-std*1.5

# calculate El Nino/La Nina masks:
ElNino_mask_timeseries=NINO34_timeseries*0+1.0
ElNino_mask_timeseries[NINO34_timeseries<ElNino_threshold]=np.nan
num_ElNino_months=np.nansum(ElNino_mask_timeseries)
LaNina_mask_timeseries=NINO34_timeseries*0+1.0
LaNina_mask_timeseries[NINO34_timeseries>LaNina_threshold]=np.nan
num_LaNina_months=np.nansum(LaNina_mask_timeseries)
print("ElNino_threshold=",ElNino_threshold, ", number of ElNino months=",num_ElNino_months
      ,"=",100*num_ElNino_months/len(NINO34_timeseries),"%");
print("LaNina_threshold=",LaNina_threshold, ", number of LaNina months=",num_LaNina_months
      ,"=",100*num_LaNina_months/len(NINO34_timeseries),"%");

fig=plt.figure(dpi=300,figsize=(6,2.5))
plt.clf()
years=np.arange(len(NINO34_timeseries))/12
plt.plot(dates,NINO34_timeseries,lw=0.5,label="NINO3.4")
plt.plot(dates,ElNino_mask_timeseries*0+ElNino_threshold,".",color="r",markersize=0.75,label="El Nino")
plt.plot(dates,LaNina_mask_timeseries*0+LaNina_threshold,".",color="b",markersize=0.75,label="La Nina")
plt.xlabel("years")
plt.ylabel("Nino 3.4 (C)")
#plt.xlim([0,1500]);
plt.legend(ncol=3)
plt.grid(lw=0.25);
plt.tight_layout()
plt.show();

#fig.savefig("Output/hurricanes-NINO34-timeseries.pdf");

In [ ]:
# calculate monthly climatologies:
if large_datasets_are_available:
    print("Calculating monthly climatologies...")
    remove_monthly_climatology=False
    for m in range(0,12):
        climatology_mask_timeseries=np.zeros(ElNino_mask_timeseries.shape)*np.nan
        climatology_mask_timeseries[m::12]=1
        monthly_climatology=np.nan
        #U200_climatology[m,:,:]   =calculate_composite(U200  ,climatology_mask_timeseries,remove_monthly_climatology,monthly_climatology)
        #V200_climatology[m,:,:]   =calculate_composite(V200  ,climatology_mask_timeseries,remove_monthly_climatology,monthly_climatology)
        #U850_climatology[m,:,:]   =calculate_composite(U200  ,climatology_mask_timeseries,remove_monthly_climatology,monthly_climatology)
        #V850_climatology[m,:,:]   =calculate_composite(V200  ,climatology_mask_timeseries,remove_monthly_climatology,monthly_climatology)
        SST_climatology[m,:,:] =calculate_composite(SST,climatology_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    print("Done calculating monthly climatologies.")

    # calculate composites for El Nina and La Nina:
    print("Calculating El Nino/ La Nina composites...")
    U200_composite_ElNino   =calculate_composite(U200  ,ElNino_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    V200_composite_ElNino   =calculate_composite(V200  ,ElNino_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    U200_composite_LaNina   =calculate_composite(U200  ,LaNina_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    V200_composite_LaNina   =calculate_composite(V200  ,LaNina_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    U850_composite_ElNino   =calculate_composite(U850  ,ElNino_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    V850_composite_ElNino   =calculate_composite(V850  ,ElNino_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    U850_composite_LaNina   =calculate_composite(U850  ,LaNina_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    V850_composite_LaNina   =calculate_composite(V850  ,LaNina_mask_timeseries,remove_monthly_climatology,monthly_climatology)
    remove_monthly_climatology=True
    SST_composite_ElNino =calculate_composite(SST,ElNino_mask_timeseries,remove_monthly_climatology,SST_climatology)
    SST_composite_LaNina =calculate_composite(SST,LaNina_mask_timeseries,remove_monthly_climatology,SST_climatology)
    print("Done calculating El Nino/ La Nina composites.")
    
    # save for use if large data are not available:
    vars2save.extend([U200_composite_ElNino,V200_composite_ElNino,U200_composite_LaNina,V200_composite_LaNina,
                      U850_composite_ElNino,V850_composite_ElNino,U850_composite_LaNina,V850_composite_LaNina,
                      SST_climatology,SST_composite_ElNino,SST_composite_LaNina])

In [ ]:
# draw the climatologies and composites:

# prepare wind shear to be plotted following Wind shear is plotted 
# by Zhu-Saravanan-Chang-2012:influence
U_EN=1.0*(U200_composite_ElNino-U850_composite_ElNino)
V_EN=1.0*(V200_composite_ElNino-V850_composite_ElNino)
U_LN=1.0*(U200_composite_LaNina-U850_composite_LaNina)
V_LN=1.0*(V200_composite_LaNina-V850_composite_LaNina)

shear_magnitude_EN=np.sqrt(U_EN**2+V_EN**2)
shear_magnitude_LN=np.sqrt(U_LN**2+V_LN**2)


# land mask:
mask=np.zeros(SST_composite_ElNino.shape)+1
mask[np.isnan(SST_composite_ElNino)]=np.nan

# add a column to lon/lat arrays to eliminate white gap at dateline:
# for atmospheric plots:
lon1=1.0*lon[-1]+lon[2]-lon[1]; lon1=np.hstack((lon,lon1))

# initialize figure:
plt.clf();
projection=ccrs.PlateCarree(central_longitude=0.0);
fig=plt.figure(figsize=(7,7.0*2.0/3.0),dpi=1200);
grid = plt.GridSpec(2, 2)#, wspace=0.4, hspace=0.3)


# El Nino:
# --------
axes=fig.add_subplot(2,2,1, projection=ccrs.PlateCarree(0))
axes.set_extent([-170, 10, -20.01, 60.01], crs=ccrs.PlateCarree(0))
axes.coastlines(resolution='110m',lw=0.3)
axes.stock_img()
plt.set_cmap('bwr')
DATA=1.0*SST_composite_ElNino[:,:]
# add a column to DATA array to eliminate white gap at dateline:
DATA1=1.0*DATA[:,0]; 
DATA1.shape=(len(DATA1[:]),1); DATA1=np.hstack((DATA,DATA1))
c=axes.pcolormesh(lon1,lat, DATA1[:,:],vmin=-3,vmax=3,shading='auto')
clb=plt.colorbar(c, shrink=0.63, pad=0.02,ax=axes)
clb.set_label('°C')
s=15
axes.quiver(lon[::s], lat[::s], U_EN[::s,::s], V_EN[::s,::s],scale=400\
            ,width=0.001,headwidth=10,headlength=10)
axes.set_title('(a) SST & shear, El Niño',pad=2,loc="left")
axes.add_patch(mpatches.Rectangle(xy=[360-80, 10], width=60, height=10
            ,edgecolor="g", lw=1, facecolor='none',transform=ccrs.PlateCarree()))
gl = axes.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='-')
gl.ylocator = mticker.FixedLocator([-20, 0, 20, 40, 60])
#gl.left_labels = True
gl.right_labels = False
gl.top_labels = False


# La Nina:
# --------
axes=fig.add_subplot(2,2,2, projection=ccrs.PlateCarree(0))#grid[0, 0])
axes.set_extent([-170, 10, -20.01, 60.01], crs=ccrs.PlateCarree())
axes.coastlines(resolution='110m',lw=0.3)
axes.stock_img()
plt.set_cmap('bwr')
DATA=1.0*SST_composite_LaNina[:,:]
# add a column to DATA array to eliminate white gap at dateline:
DATA1=1.0*DATA[:,0]; 
DATA1.shape=(len(DATA1[:]),1); DATA1=np.hstack((DATA,DATA1))
c=axes.pcolormesh(lon1,lat, DATA1[:,:],vmin=-3,vmax=3,shading='auto')
clb=plt.colorbar(c, shrink=0.63, pad=0.02,ax=axes)
clb.set_label('°C')
s=15
axes.quiver(lon[::s], lat[::s], U_LN[::s,::s], V_LN[::s,::s],scale=400\
            ,width=0.001,headwidth=10,headlength=10)
axes.set_title('(b) SST & shear, La Niña',pad=6,loc="left")
axes.add_patch(mpatches.Rectangle(xy=[360-80, 10], width=60, height=10
            ,edgecolor="g", lw=1, facecolor='none',transform=ccrs.PlateCarree()))
gl = axes.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='-')
gl.left_labels = False
gl.right_labels = False
gl.top_labels = False


# El Nino minus La Nina:
# ----------------------
axes=fig.add_subplot(2,2,(3,4), projection=ccrs.PlateCarree(0))#grid[0, 0])
axes.set_extent([-170, 10, -20.01, 60.1], crs=ccrs.PlateCarree())
axes.coastlines(resolution='110m',lw=0.3)
axes.stock_img()
plt.set_cmap('bwr')
c=axes.pcolormesh(lon1,lat, (shear_magnitude_EN-shear_magnitude_LN)*mask\
                  ,vmin=-15,vmax=15,shading='auto')
clb=plt.colorbar(c, shrink=0.95, pad=0.02,ax=axes)
clb.set_label("m/s")
s=15
U=U_EN[::s,::s]-U_LN[::s,::s]
V=V_EN[::s,::s]-V_LN[::s,::s]
axes.quiver(lon[::s], lat[::s], U, V,scale=400,width=0.001\
            ,headwidth=10,headlength=10)
axes.set_title('(c) Shear, El Niño $-$ La Niña',pad=2,loc="left")
axes.add_patch(mpatches.Rectangle(xy=[360-80, 10], width=60, height=10
        ,edgecolor="g", lw=1, facecolor='none', transform=ccrs.PlateCarree()))
gl = axes.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', linestyle='-')
gl.ylocator = mticker.FixedLocator([-20, 0, 20, 40, 60])
gl.right_labels = False
gl.top_labels = False

# finalize and show plot:
plt.subplots_adjust(top=0.92, bottom=0.08, left=0.05, right=0.99\
                    ,hspace=0.07,wspace=0.01);
#move bottom panel and its colorbar to left a bit:
box = axes.get_position()
box.x0 = box.x0 - 0.05
box.x1 = box.x1 - 0.05
axes.set_position(box)
cbox=clb.ax.get_position()
cbox.x0 = cbox.x0 - 0.05
cbox.x1 = cbox.x1 - 0.05
clb.ax.set_position(cbox)
plt.show();
filename="hurricanes-SST-and-jet-shear-ENSO-composites-ERA5"
fig.savefig("Output/"+filename+".png"
            ,bbox_inches="tight", pad_inches=0.02)
png = "Output/"+filename+".png"
pdf = "Output/"+filename+".pdf"
subprocess.run(["magick", png, pdf], check=True);

In [ ]:
# pickle all vars to save:
def pickle_vars(fileName, env, variables):
    d = dict([(x, env[x]) for v in variables for x in env if v is env[x]])
    with open(fileName, 'wb') as f:
        pickle.dump(d, f)

if large_datasets_are_available:
    pickle_vars('Output/hurricanes_ElNino_LaNina_jet_composites_ERA5.pickle',locals(),vars2save)

print("done.")